# COPY INTO COMMAND
#### Incrementally loads data into delta lake tables from cloud storage
#### Supports schema evolution
#### Supports wide range of file format
#### Alternative to Auto Loader for Batch Ingestion

In [0]:
%sql
CREATE TABLE IF NOT EXISTS demo.delta_lake.raw_stock_prices;

### Incrementally Load New files into the table

In [0]:
DELETE from demo.delta_lake.raw_stock_prices;

In [0]:
%sql
COPY INTO demo.delta_lake.raw_stock_prices
FROM 'abfss://demo@oosstorage2.dfs.core.windows.net/landing/stock_prices'
FILEFORMAT = JSON
FORMAT_OPTIONS ('inferSchema' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true');

In [0]:
%sql
DESC HISTORY demo.delta_lake.raw_stock_prices;

In [0]:
%sql
SELECT * from demo.delta_lake.raw_stock_prices;

###MERGE STATEMENT in DELTA LAKE

In [0]:
%sql
CREATE TABLE IF NOT EXISTS demo.delta_lake.stock_prices
(stock_id STRING,
price DOUBLE,
trading_date DATE   
)

In [0]:
%sql
MERGE INTO demo.delta_lake.stock_prices as target
USING demo.delta_lake.raw_stock_prices as source
ON source.stock_id = target.stock_id
WHEN MATCHED AND source.status = 'ACTIVE' THEN 
UPDATE SET target.price = source.price, target.trading_date = source.trading_date
WHEN MATCHED AND source.status = 'DELISTED' THEN
DELETE
WHEN NOT MATCHED AND source.status = 'ACTIVE' THEN 
INSERT (stock_id, price, trading_date) VALUES (source.stock_id,source.price,source.trading_date);

In [0]:
select * from demo.delta_lake.stock_prices;

In [0]:
DESC HISTORY demo.delta_lake.stock_prices;